# Notebook 03 - LSTM: Pre-Training & Fine-Tuning

Phase 1: Pre-Training auf synthetischen Daten (Notebook 01)
Phase 2: Fine-Tuning auf echten interpolierten Daten (Notebook 02)
Phase 3 (optional): Ablationsvergleich ohne Pre-Training

**Wichtig:** Fuer Phase 2 werden nur Planungsraeume mit VOLLSTAENDIGER
Zeitreihe verwendet (alle Anker-Jahre aus meta.pkl vorhanden). plr_id
wird konsequent auf 8-stellige Strings normalisiert (zfill), da CSV-
Rundtrips fuehrende Nullen verschlucken koennen.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 1: Imports & gemeinsame Klassen
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

with open('../models/meta.pkl', 'rb') as f:
    meta = pickle.load(f)
SEQ_LEN = meta.get('seq_len', 3)
ALLE_JAHRE_META = [int(y) for y in meta.get('jahre', [])]
print(f'✅ Meta geladen: SEQ_LEN={SEQ_LEN}, Anker-Jahre={ALLE_JAHRE_META}')


class BerlinDataset(Dataset):
    def __init__(self, df_full, feature_cols, target_col, seq_len=3, target_years=None):
        self.sequences, self.targets, self.plr_ids = [], [], []
        if target_years is not None:
            target_years = set(int(y) for y in target_years)
        for plr_id, group in df_full.groupby('plr_id'):
            group = group.sort_values('year').reset_index(drop=True)
            X = group[feature_cols].values.astype(np.float32)
            y = group[target_col].values.astype(np.float32)
            yrs = group['year'].values
            for i in range(len(group) - seq_len):
                target_yr = int(yrs[i + seq_len])
                if target_years is None or target_yr in target_years:
                    self.sequences.append(X[i:i+seq_len])
                    self.targets.append(y[i+seq_len])
                    self.plr_ids.append(str(plr_id))
        if len(self.sequences) > 0:
            self.sequences = torch.FloatTensor(np.array(self.sequences))
            self.targets   = torch.FloatTensor(np.array(self.targets))
        else:
            self.sequences = torch.zeros(0, seq_len, len(feature_cols))
            self.targets   = torch.zeros(0)
    def __len__(self): return len(self.targets)
    def __getitem__(self, idx): return self.sequences[idx], self.targets[idx]


class BerlinLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            dropout=dropout if num_layers>1 else 0, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 16), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(16, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def train_loop(model, train_dl, val_dl, lr, max_epochs, patience,
               save_path, label='Training'):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    best_val, counter = float('inf'), 0
    train_losses, val_losses = [], []

    print(f'🚀 {label} startet... (lr={lr}, max_epochs={max_epochs})')
    for epoch in range(max_epochs):
        model.train()
        t_loss = 0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X).squeeze(-1), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            t_loss += loss.item()

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for X, y in val_dl:
                X, y = X.to(device), y.to(device)
                v_loss += criterion(model(X).squeeze(-1), y).item()

        t_loss /= max(len(train_dl), 1)
        v_loss /= max(len(val_dl), 1)
        train_losses.append(t_loss); val_losses.append(v_loss)
        scheduler.step(v_loss)

        if v_loss < best_val:
            best_val, counter = v_loss, 0
            torch.save(model.state_dict(), save_path)
        else:
            counter += 1
        if counter >= patience:
            print(f'⏹️ Early Stopping bei Epoche {epoch}')
            break
        if epoch % 10 == 0:
            print(f'Epoch {epoch:3d} | Train: {t_loss:.4f} | Val: {v_loss:.4f}')

    print(f'✅ {label} abgeschlossen! Bester Val-Loss: {best_val:.4f}\n')
    return train_losses, val_losses, best_val

print('✅ Klassen & Funktionen definiert!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 1: PRE-TRAINING auf synthetischen Daten
# ═══════════════════════════════════════════════════════════════

df_synth = pd.read_csv('../data/raw/mss_pretraining_synthetisch.csv')
df_synth['plr_id'] = df_synth['plr_id'].astype(str).str.zfill(8)

FEATURE_COLS_SYNTH = [
    'arbeitslosigkeit', 'sgb2_quote', 'kinderarmut',
    'dist_zentrum', 'status_lag'
]
TARGET = 'status_next'

years_synth = sorted(df_synth['year'].unique())
train_years_synth = years_synth[:-4]
val_years_synth   = years_synth[-4:-2]

pretrain_scaler = StandardScaler()
df_synth_scaled = df_synth.copy()
mask_train = df_synth['year'].isin(train_years_synth)
pretrain_scaler.fit(df_synth.loc[mask_train, FEATURE_COLS_SYNTH])
df_synth_scaled[FEATURE_COLS_SYNTH] = pretrain_scaler.transform(df_synth[FEATURE_COLS_SYNTH])

pretrain_train_set = BerlinDataset(df_synth_scaled, FEATURE_COLS_SYNTH, TARGET,
                                    SEQ_LEN, target_years=train_years_synth)
pretrain_val_set   = BerlinDataset(df_synth_scaled, FEATURE_COLS_SYNTH, TARGET,
                                    SEQ_LEN, target_years=val_years_synth)

pretrain_train_dl = DataLoader(pretrain_train_set, batch_size=64, shuffle=True)
pretrain_val_dl   = DataLoader(pretrain_val_set,   batch_size=64, shuffle=False)

print(f'Pre-Training Sequenzen: Train={len(pretrain_train_set)} Val={len(pretrain_val_set)}')

if len(pretrain_train_set) == 0:
    raise RuntimeError(
        f'Pre-Training Train-Set ist leer! SEQ_LEN={SEQ_LEN} zu gross fuer '
        f'synthetische Jahre. Notebook 01 hat 24 Jahre - sollte reichen, '
        'bitte Notebook 01 Ausgabe pruefen.'
    )

lstm_pretrain = BerlinLSTM(input_size=len(FEATURE_COLS_SYNTH)).to(device)

pt_train_losses, pt_val_losses, pt_best = train_loop(
    lstm_pretrain, pretrain_train_dl, pretrain_val_dl,
    lr=0.001, max_epochs=150, patience=15,
    save_path='../models/lstm_pretrained.pt', label='PRE-TRAINING (synthetisch)'
)

with open('../models/pretrain_scaler.pkl', 'wb') as f:
    pickle.dump(pretrain_scaler, f)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 2a: Echte Daten laden & auf VOLLSTAENDIGE Zeitreihen filtern
# ═══════════════════════════════════════════════════════════════

train   = pd.read_csv('../data/final/train.csv')
train['plr_id'] = train['plr_id'].astype(str).str.zfill(8)
val     = pd.read_csv('../data/final/val.csv')
val['plr_id'] = val['plr_id'].astype(str).str.zfill(8)
test    = pd.read_csv('../data/final/test.csv')
test['plr_id'] = test['plr_id'].astype(str).str.zfill(8)
df_full_raw = pd.read_csv('../data/final/df_full.csv')
df_full_raw['plr_id'] = df_full_raw['plr_id'].astype(str).str.zfill(8)

with open('../models/feature_cols.pkl', 'rb') as f:
    FEATURE_COLS = pickle.load(f)
with open('../models/target.pkl', 'rb') as f:
    TARGET = pickle.load(f)

print(f'ECHTE Daten (roh): Train={len(train)} Val={len(val)} Test={len(test)}')
print(f'Features: {FEATURE_COLS}')

required_years = set(ALLE_JAHRE_META) if ALLE_JAHRE_META else set(df_full_raw['year'].unique())
jahre_pro_plr = df_full_raw.groupby('plr_id')['year'].apply(lambda s: set(int(y) for y in s))
vollstaendige_plr = jahre_pro_plr[jahre_pro_plr.apply(lambda s: required_years.issubset(s))].index
df_full = df_full_raw[df_full_raw['plr_id'].isin(vollstaendige_plr)].copy()

print(f'\n📊 Vollstaendigkeits-Check:')
print(f'   Benoetigte Anker-Jahre: {sorted(required_years)}')
print(f'   PLR gesamt:             {df_full_raw["plr_id"].nunique()}')
print(f'   PLR mit vollstaendiger Zeitreihe: {len(vollstaendige_plr)}')
if len(vollstaendige_plr) < df_full_raw['plr_id'].nunique() * 0.5:
    print(f'   ⚠️  Mehr als die Haelfte der PLR wurde ausgeschlossen - '
          f'Areal-Interpolation in Notebook 02 pruefen!')

if FEATURE_COLS != FEATURE_COLS_SYNTH:
    print('\n⚠️  ACHTUNG: Feature-Listen unterscheiden sich!')
    print(f'   Synthetisch: {FEATURE_COLS_SYNTH}')
    print(f'   Echt:        {FEATURE_COLS}')
    FEATURE_COLS_COMMON = [f for f in FEATURE_COLS if f in FEATURE_COLS_SYNTH]
    print(f'   Verwende gemeinsame Features ({len(FEATURE_COLS_COMMON)}): {FEATURE_COLS_COMMON}')
else:
    FEATURE_COLS_COMMON = FEATURE_COLS
    print('✅ Feature-Listen stimmen ueberein!')

TRAIN_YEARS = [int(y) for y in train['year'].unique()]
VAL_YEARS   = [int(y) for y in val['year'].unique()]
TEST_YEARS  = [int(y) for y in test['year'].unique()]

print(f'\nTrain Jahre: {sorted(TRAIN_YEARS)}')
print(f'Val Jahre:   {sorted(VAL_YEARS)}')
print(f'Test Jahre:  {sorted(TEST_YEARS)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 2b: Datasets bauen (auf gefiltertem df_full)
# ═══════════════════════════════════════════════════════════════

ft_train_set = BerlinDataset(df_full, FEATURE_COLS_COMMON, TARGET, SEQ_LEN, TRAIN_YEARS)
ft_val_set   = BerlinDataset(df_full, FEATURE_COLS_COMMON, TARGET, SEQ_LEN, VAL_YEARS)
ft_test_set  = BerlinDataset(df_full, FEATURE_COLS_COMMON, TARGET, SEQ_LEN, TEST_YEARS)

print(f'Fine-Tuning Sequenzen: Train={len(ft_train_set)} Val={len(ft_val_set)} Test={len(ft_test_set)}')

if len(ft_train_set) == 0 or len(ft_val_set) == 0:
    raise RuntimeError(
        f'Train oder Val Set ist leer! (Train={len(ft_train_set)}, Val={len(ft_val_set)}) '
        f'SEQ_LEN={SEQ_LEN}, benoetigte Jahre={sorted(required_years)}. '
        'Bitte pruefen ob Notebook 02 korrekt und vollstaendig durchgelaufen ist.'
    )

print('✅ Alle Splits haben gueltige Sequenzen!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Fine-Tuning durchfuehren (mit Pre-Training-Gewichten als Start)
# ═══════════════════════════════════════════════════════════════

ft_train_dl = DataLoader(ft_train_set, batch_size=16, shuffle=True)
ft_val_dl   = DataLoader(ft_val_set, batch_size=min(16, max(len(ft_val_set),1)), shuffle=False)

lstm_finetuned = BerlinLSTM(input_size=len(FEATURE_COLS_COMMON)).to(device)

if FEATURE_COLS_COMMON == FEATURE_COLS_SYNTH:
    lstm_finetuned.load_state_dict(
        torch.load('../models/lstm_pretrained.pt', map_location=device))
    print('✅ Pre-Training-Gewichte geladen!')
else:
    print('⚠️ Feature-Anzahl unterschiedlich - starte ohne Pre-Training-Gewichte '
          '(nur zufaellige Initialisierung)')

ft_train_losses, ft_val_losses, ft_best = train_loop(
    lstm_finetuned, ft_train_dl, ft_val_dl,
    lr=0.0001, max_epochs=50, patience=10,
    save_path='../models/lstm_finetuned.pt', label='FINE-TUNING (echte Daten)'
)

with open('../models/lstm_feature_cols_common.pkl', 'wb') as f:
    pickle.dump(FEATURE_COLS_COMMON, f)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 3 (optional): Ablationsvergleich - OHNE Pre-Training
# ═══════════════════════════════════════════════════════════════

lstm_scratch = BerlinLSTM(input_size=len(FEATURE_COLS_COMMON)).to(device)

scratch_train_losses, scratch_val_losses, scratch_best = train_loop(
    lstm_scratch, ft_train_dl, ft_val_dl,
    lr=0.001, max_epochs=50, patience=10,
    save_path='../models/lstm_scratch.pt', label='ABLATION (ohne Pre-Training)'
)

print('📊 Vergleich Best Val-Loss:')
print(f'   Mit Pre-Training:    {ft_best:.4f}')
print(f'   Ohne Pre-Training:   {scratch_best:.4f}')
verbesserung = (1 - ft_best/scratch_best) * 100 if scratch_best > 0 else 0
print(f'   Verbesserung durch Pre-Training: {verbesserung:.1f}%')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Lernkurven aller 3 Phasen visualisieren
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(pt_train_losses, label='Train', color='steelblue')
axes[0].plot(pt_val_losses, label='Val', color='red')
axes[0].set_title('Phase 1: Pre-Training\n(synthetisch)', fontweight='bold')
axes[0].set_xlabel('Epoche'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ft_train_losses, label='Train', color='steelblue')
axes[1].plot(ft_val_losses, label='Val', color='red')
axes[1].set_title('Phase 2: Fine-Tuning\n(echte Daten)', fontweight='bold')
axes[1].set_xlabel('Epoche'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(scratch_train_losses, label='Train', color='steelblue')
axes[2].plot(scratch_val_losses, label='Val', color='red')
axes[2].set_title('Phase 3: Ablation\n(ohne Pre-Training)', fontweight='bold')
axes[2].set_xlabel('Epoche'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('LSTM: Lernkurven aller Trainingsphasen', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/03_lstm_lernkurven_alle_phasen.png', dpi=150)
plt.show()

print('✅ NOTEBOOK 03 ABGESCHLOSSEN!')
print('\n📁 Gespeicherte Modelle:')
print('   models/lstm_pretrained.pt   (Phase 1)')
print('   models/lstm_finetuned.pt    (Phase 2 - FINALES Modell)')
print('   models/lstm_scratch.pt      (Phase 3 - Ablationsvergleich)')
print('   models/lstm_feature_cols_common.pkl (verwendete Features)')
print('\n⚠️  Fuer SHAP (Notebook 06) NUR lstm_finetuned.pt verwenden!')
print('\n→ Weiter mit Notebook 04 (TFT Pre-Training + Fine-Tuning)')